# 神经网络的学习练习参考答案（第 4 章）

本文件是配套练习题的参考答案，请先在手写练习后再对照。

# 神经网络的学习练习（第 4 章）



- 损失函数（MSE、交叉熵误差、BCELoss、CrossEntropyLoss、L1Loss、SmoothL1Loss）
- 随机梯度下降法（梯度下降、Epoch / Batch Size / Iteration、optim.SGD）
- 数据集的创建和分批（Dataset、TensorDataset、DataLoader）
- 反向传播算法（计算图、链式法则、加法节点、乘法节点）
- 神经网络的反向传播（ReLU、Sigmoid、全连接层）
- PyTorch 的自动微分模块（requires_grad、backward、is_leaf、detach）
- 应用案例（线性回归、手写数字识别）

使用方法：阅读每道题的描述后，在下方代码单元格中**手写代码**完成练习；写完后可与 `answer` 目录下的答案对照。

部分练习所需的数据位于项目根目录的 `data/` 下，本 notebook 中使用相对路径 `../../data/...`。

先执行下面的单元格导入所需的库。

In [ ]:
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from torch import optim
from torch.utils.data import Dataset, TensorDataset, DataLoader

print("torch version:", torch.__version__)

## 4.1 损失函数

**练习 1**：手写均方误差（MSE）并与 `nn.MSELoss` 对比

给定预测值 `y_pred = torch.tensor([2.5, 0.0, 2.1, 7.8])`、真实值 `y_true = torch.tensor([3.0, -0.5, 2.0, 8.0])`：

1. 按公式 $L=\frac{1}{n}\sum_{i=1}^{n}(y_i-t_i)^2$ 手写计算 MSE
2. 用 `nn.MSELoss()` 计算一次
3. 验证两者结果一致

提示：`y_pred - y_true` 得到误差向量，`** 2` 求平方，`.mean()` 求平均。

In [ ]:
# 练习 1：手写均方误差并与 nn.MSELoss 对比

# 练习 1：手写均方误差（MSE）并与 nn.MSELoss 对比

y_pred = torch.tensor([2.5, 0.0, 2.1, 7.8])
y_true = torch.tensor([3.0, -0.5, 2.0, 8.0])

# 1. 手写 MSE：先求误差平方，再取平均
mse_manual = ((y_pred - y_true) ** 2).mean()
print("手写 MSE    :", mse_manual.item())

# 2. 使用 nn.MSELoss
loss_fn = nn.MSELoss()
mse_torch = loss_fn(y_pred, y_true)
print("nn.MSELoss  :", mse_torch.item())

# 3. 验证一致
print("两者是否一致:", torch.allclose(mse_manual, mse_torch))

**练习 2**：手写交叉熵误差（one-hot 标签）

给定 3 个样本、3 个类别的预测得分 `logits = torch.tensor([[2.0, 1.0, 0.1], [0.5, 2.5, 0.3], [0.1, 0.2, 3.0]])`，
以及真实标签的 one-hot 编码 `t = torch.tensor([[1., 0., 0.], [0., 1., 0.], [0., 0., 1.]])`：

1. 先对 `logits` 在 `dim=1` 上做 softmax 得到概率 `y`
2. 按公式 $L=-\frac{1}{n}\sum_{i} t_i \log y_i$ 手写计算交叉熵误差
3. 用 `nn.CrossEntropyLoss()` 传入 `logits` 与标签索引 `torch.tensor([0, 1, 2])` 计算，验证两者一致

提示：`-(t * torch.log(y)).sum(dim=1).mean()`；`CrossEntropyLoss` 内部相当于 `LogSoftmax` + `NLLLoss`，因此直接传 logits。

In [ ]:
# 练习 2：手写交叉熵误差

# 练习 2：手写交叉熵误差（one-hot 标签）

logits = torch.tensor([[2.0, 1.0, 0.1], [0.5, 2.5, 0.3], [0.1, 0.2, 3.0]])
t = torch.tensor([[1., 0., 0.], [0., 1., 0.], [0., 0., 1.]])

# 1. softmax 得到概率
y = torch.softmax(logits, dim=1)

# 2. 手写交叉熵误差：只保留正确类别对应的 -log(y)
ce_manual = -(t * torch.log(y)).sum(dim=1).mean()
print("手写交叉熵  :", ce_manual.item())

# 3. 使用 nn.CrossEntropyLoss（直接传入 logits 与标签索引）
loss_fn = nn.CrossEntropyLoss()
ce_torch = loss_fn(logits, torch.tensor([0, 1, 2]))
print("CrossEntropy:", ce_torch.item())

print("两者是否一致:", torch.allclose(ce_manual, ce_torch))

**练习 3**：二分类任务的 BCELoss

1. 用 `torch.randn((4, 1))` 生成 4 个样本的原始输出 `logits`
2. 用 `torch.sigmoid` 得到预测概率 `pred`（表示样本为 1 的概率）
3. 真实标签 `target = torch.tensor([[1.], [0.], [1.], [0.]], dtype=torch.float32)`
4. 用 `nn.BCELoss()` 计算损失
5. 思考并说明：为什么 `BCELoss` 的输入必须先经过 Sigmoid？

In [ ]:
# 练习 3：二分类任务的 BCELoss

# 练习 3：二分类任务的 BCELoss

torch.manual_seed(42)

# 1. 原始输出
logits = torch.randn((4, 1))
# 2. 经过 Sigmoid 得到概率
pred = torch.sigmoid(logits)
print("预测概率:\n", pred)

# 3. 真实标签
target = torch.tensor([[1.], [0.], [1.], [0.]], dtype=torch.float32)
print("真实标签:\n", target)

# 4. 计算二分类交叉熵损失
loss = nn.BCELoss()
print("BCELoss:", loss(pred, target).item())

# 5. BCELoss 的公式为 -(y*log(p) + (1-y)*log(1-p))，要求 p 是 (0, 1) 区间内的概率值，
#    而网络的原始输出 logits 取值范围是全体实数，因此必须先经过 Sigmoid 压缩到 (0, 1)。

**练习 4**：多分类任务的 CrossEntropyLoss

1. 真实值为类别标签：`target = torch.tensor([1, 0, 3, 2, 5, 4])`，预测 `input = torch.randn((6, 8))`，用 `nn.CrossEntropyLoss()` 计算损失
2. 真实值为概率分布：`target = torch.randn(6, 8).softmax(dim=1)`，预测 `input = torch.randn((6, 8))`，同样计算损失
3. 回答：为什么使用 `CrossEntropyLoss` 时，网络最后一层不需要再加 Softmax？

In [ ]:
# 练习 4：多分类任务的 CrossEntropyLoss

# 练习 4：多分类任务的 CrossEntropyLoss

torch.manual_seed(42)
loss_fn = nn.CrossEntropyLoss()

# 1. 真实值为类别标签（常用形式）
target = torch.tensor([1, 0, 3, 2, 5, 4])
input = torch.randn((6, 8))
print("标签形式 loss:", loss_fn(input, target).item())

# 2. 真实值为概率分布
target = torch.randn(6, 8).softmax(dim=1)
input = torch.randn((6, 8))
print("概率形式 loss:", loss_fn(input, target).item())

# 3. CrossEntropyLoss 内部等价于 LogSoftmax + NLLLoss，
#    它会对输入的 logits 先做 softmax 再做对数运算，因此网络最后一层只需输出 logits，
#    不需要（也不能）额外加 Softmax，否则相当于做了两次归一化。

**练习 5**：回归损失函数对比（L1 / L2 / Smooth L1）

1. 用 `torch.manual_seed(42)` 固定随机种子，生成 `input = torch.randn(5)`、`target = torch.randn(5)`
2. 分别用 `nn.L1Loss()`、`nn.MSELoss()`、`nn.SmoothL1Loss()` 计算损失并打印
3. 把 `target` 的最后一个元素改成 `100.0`（模拟异常值），重新计算三种损失，
   观察哪种损失对异常值最敏感（提示：L2 Loss 对异常值敏感，遇到异常值易梯度爆炸）

In [ ]:
# 练习 5：回归损失函数对比（L1 / L2 / Smooth L1）

# 练习 5：回归损失函数对比（L1 / L2 / Smooth L1）

torch.manual_seed(42)
input = torch.randn(5)
target = torch.randn(5)

l1 = nn.L1Loss()
l2 = nn.MSELoss()
smooth = nn.SmoothL1Loss()

print("--- 无异常值 ---")
print("L1Loss       :", l1(input, target).item())
print("MSELoss      :", l2(input, target).item())
print("SmoothL1Loss :", smooth(input, target).item())

# 3. 把最后一个目标值改成 100.0，模拟异常值
target_outlier = target.clone()
target_outlier[-1] = 100.0

print("--- 含异常值 ---")
print("L1Loss       :", l1(input, target_outlier).item())
print("MSELoss      :", l2(input, target_outlier).item())
print("SmoothL1Loss :", smooth(input, target_outlier).item())
# 对比可见 MSELoss 因为对误差取平方，异常值会让损失急剧变大；L1Loss 对异常值最鲁棒；
# SmoothL1Loss 在误差小的时候表现得像 L2、误差大时表现得像 L1，兼顾平滑与鲁棒。

## 4.2 随机梯度下降法

**练习 6**：手写梯度下降法

用梯度下降法求函数 $f(x)=(x-3)^2$ 的最小值点，其导数为 $f'(x)=2(x-3)$。

1. 初始值取 `x = 10.0`
2. 学习率 `lr = 0.1`，迭代 30 次，按 $x = x - \eta f'(x)$ 更新
3. 每 5 次打印一次 `x` 的值，观察是否逐渐逼近 3.0

In [ ]:
# 练习 6：手写梯度下降法求最小值

# 练习 6：手写梯度下降法

x = 10.0        # 初始值
lr = 0.1        # 学习率

for i in range(30):
    grad = 2 * (x - 3)      # f'(x) = 2(x - 3)
    x = x - lr * grad       # 沿负梯度方向更新
    if (i + 1) % 5 == 0:
        print(f"第 {i + 1:2d} 次迭代: x = {x:.6f}")

print("最终 x =", x, "（理论最小值点 x = 3）")

**练习 7**：Epoch / Batch Size / Iteration 的计算

数据集共有 2000 个样本，训练 10 个 Epoch，Batch Size 设为 64（最后一个 batch 不足 64 也算一个）。

1. 计算每个 Epoch 有多少个 Batch
2. 计算每个 Epoch 的迭代次数（Iteration）
3. 计算总迭代次数与总训练样本数
4. 打印上述结果

In [ ]:
# 练习 7：Epoch / Batch Size / Iteration 的计算

# 练习 7：Epoch / Batch Size / Iteration 的计算

total_samples = 2000
epochs = 10
batch_size = 64

# 1. 每个 Epoch 的 Batch 个数（向上取整）
num_batches = total_samples // batch_size + (1 if total_samples % batch_size else 0)
# 2. 每个 Epoch 的迭代次数等于 Batch 个数
iter_per_epoch = num_batches
# 3. 总迭代次数与总训练样本数
total_iter = epochs * iter_per_epoch
total_trained = epochs * total_samples

print(f"每个 Epoch 的 Batch 个数 : {num_batches}")
print(f"每个 Epoch 的迭代次数    : {iter_per_epoch}")
print(f"总迭代次数               : {total_iter}")
print(f"总训练样本数             : {total_trained}")

**练习 8**：使用 optim.SGD 更新参数

1. 定义可训练权重 `w = torch.tensor([[1.0, 2.0]], requires_grad=True)`，
   输入 `x = torch.tensor([[3.0, 4.0]])`，目标值 `target = torch.tensor([[10.0]])`
2. 前向传播 `y = x @ w.T`，用 `nn.MSELoss()` 计算损失
3. 用 `optim.SGD([w], lr=0.01)` 创建优化器，依次执行 `zero_grad()` → `backward()` → `step()`
4. 打印更新前的 `w`、`w.grad` 与更新后的 `w`

In [ ]:
# 练习 8：使用 optim.SGD 更新参数

# 练习 8：使用 optim.SGD 更新参数

w = torch.tensor([[1.0, 2.0]], requires_grad=True)
x = torch.tensor([[3.0, 4.0]])
target = torch.tensor([[10.0]])

# 1. 前向传播 + 计算损失
y = x @ w.T
loss_fn = nn.MSELoss()
loss_value = loss_fn(y, target)
print("预测值 y =", y.item(), " 损失 =", loss_value.item())

# 2. 创建优化器
optimizer = optim.SGD([w], lr=0.01)

# 3. 清空梯度 -> 反向传播 -> 更新参数
optimizer.zero_grad()
loss_value.backward()
print("更新前 w =", w.data, " w.grad =", w.grad)
optimizer.step()
print("更新后 w =", w.data)
# 梯度方向是损失增大的方向，沿负梯度更新后 w 变小，预测值 y 因此变小、更接近目标值 10

## 4.3 数据集的创建和分批

**练习 9**：自定义 Dataset

继承 `torch.utils.data.Dataset` 实现 `SimpleDataset`：

1. `__init__` 保存传入的数据
2. `__len__` 返回数据集大小
3. `__getitem__` 按索引返回单个样本
4. 用 `data = [1, 2, 3, 4, 5]` 创建数据集，打印 `dataset[0]`、`dataset[2]` 以及 `len(dataset)`

In [ ]:
# 练习 9：自定义 Dataset

# 练习 9：自定义 Dataset

class SimpleDataset(Dataset):
    # 初始化
    def __init__(self, data):
        self.data = data

    # 返回数据集大小
    def __len__(self):
        return len(self.data)

    # 按索引获取单个样本
    def __getitem__(self, index):
        return self.data[index]


data = [1, 2, 3, 4, 5]
dataset = SimpleDataset(data)

print("dataset[0] =", dataset[0])
print("dataset[2] =", dataset[2])
print("len(dataset) =", len(dataset))

**练习 10**：TensorDataset 与 DataLoader 分批

1. 定义 `X = torch.randn(10, 3)`、`y = torch.randn(10)`
2. 用 `TensorDataset(X, y)` 构建数据集，打印第 1 个样本
3. 用 `DataLoader(dataset, batch_size=2, shuffle=False)` 分批遍历，打印每个 batch 中 `x_batch`、`y_batch` 的形状
4. 统计一共分了多少个 batch

In [ ]:
# 练习 10：TensorDataset 与 DataLoader 分批

# 练习 10：TensorDataset 与 DataLoader 分批

torch.manual_seed(42)
X = torch.randn(10, 3)
y = torch.randn(10)

# 1. 构建数据集
dataset = TensorDataset(X, y)
print("第 1 个样本:", dataset[0])

# 2. 创建数据加载器
dataloader = DataLoader(dataset, batch_size=2, shuffle=False)

# 3. 分批遍历
count = 0
for x_batch, y_batch in dataloader:
    count += 1
    print(f"batch {count}: x_batch {tuple(x_batch.shape)}, y_batch {tuple(y_batch.shape)}")

print("总 batch 个数:", count)
# shuffle=True 时每个 Epoch 会打乱数据顺序，有助于提升泛化能力

## 4.4 反向传播算法

**练习 11**：用链式法则计算复合函数的偏导数

对于复合函数 $z=(x+y)^2$，令 $u=x+y$，则

$$\frac{\partial z}{\partial x}=\frac{\partial z}{\partial u}\frac{\partial u}{\partial x}=2u \times 1=2(x+y)$$

1. 取 `x = 2.0`、`y = 3.0`，按解析公式计算 $\frac{\partial z}{\partial x}$ 与 $\frac{\partial z}{\partial y}$
2. 用 PyTorch 的自动微分（`requires_grad=True` + `z.backward()`）计算 `x.grad`、`y.grad`
3. 验证两者结果一致

In [ ]:
# 练习 11：用链式法则计算复合函数的偏导数

# 练习 11：用链式法则计算复合函数的偏导数

# 1. 按解析公式计算（z = (x+y)^2，dz/dx = dz/dy = 2(x+y)）
x_value, y_value = 2.0, 3.0
dz_dx_manual = 2 * (x_value + y_value)
dz_dy_manual = 2 * (x_value + y_value)
print("解析结果 dz/dx =", dz_dx_manual, " dz/dy =", dz_dy_manual)

# 2. 用自动微分计算
x = torch.tensor(x_value, requires_grad=True)
y = torch.tensor(y_value, requires_grad=True)
u = x + y
z = u ** 2
z.backward()

print("自动微分 x.grad =", x.grad.item(), " y.grad =", y.grad.item())

# 3. 验证一致
print("dz/dx 是否一致:", torch.allclose(x.grad, torch.tensor(dz_dx_manual)))
print("dz/dy 是否一致:", torch.allclose(y.grad, torch.tensor(dz_dy_manual)))

**练习 12**：加法节点与乘法节点的反向传播

1. 加法节点：取 `x = 2.0`、`y = 3.0`，计算 `z = x + y` 并反向传播，
   验证 $\frac{\partial z}{\partial x}=\frac{\partial z}{\partial y}=1$（加法把上游梯度原样向下游传递）
2. 乘法节点：取 `x = 2.0`、`y = 3.0`，计算 `z = x * y` 并反向传播，
   验证 $\frac{\partial z}{\partial x}=y$、$\frac{\partial z}{\partial y}=x$（乘法把上游梯度乘以输入的翻转值传递）
3. 打印结果并说明两种节点反向传播方式的区别

In [ ]:
# 练习 12：加法节点与乘法节点的反向传播

# 练习 12：加法节点与乘法节点的反向传播

# 1. 加法节点
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)
z = x + y
z.backward()
print(f"加法: dz/dx = {x.grad.item()} (应为 1), dz/dy = {y.grad.item()} (应为 1)")

# 2. 乘法节点
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)
z = x * y
z.backward()
print(f"乘法: dz/dx = {x.grad.item()} (应为 y = 3), dz/dy = {y.grad.item()} (应为 x = 2)")

# 加法节点的反向传播把上游传来的梯度原样传递给两个输入；
# 乘法节点则把上游梯度乘以另一个输入（交换输入）后传递。

## 4.5 神经网络的反向传播

**练习 13**：手写 ReLU 的反向传播

ReLU 函数 $f(x)=\max(0,x)$，其导数为：$x \le 0$ 时为 0，$x > 0$ 时为 1。

1. 实现 `ReLU` 类：
   - `forward`：记录 `mask = (x <= 0)`，并把 `mask` 位置的输出置 0
   - `backward`：把上游梯度 `dout` 在 `mask` 位置置 0 后返回（因为那些位置导数为 0）
2. 输入 `x = torch.tensor([-2., -1., 0., 1., 2.])`，做前向与反向计算
3. 观察反向结果，只有正数位置的梯度被保留

In [ ]:
# 练习 13：手写 ReLU 的反向传播

# 练习 13：手写 ReLU 的反向传播

class ReLU:
    def __init__(self):
        self.mask = None

    def forward(self, x):
        # 记录 x <= 0 的位置（这些位置导数为 0）
        self.mask = (x <= 0)
        out = x.clone()
        out[self.mask] = 0
        return out

    def backward(self, dout):
        # 正向传播时输入 <= 0 的位置，反向传播时梯度置 0
        dout = dout.clone()
        dout[self.mask] = 0
        return dout


x = torch.tensor([-2., -1., 0., 1., 2.])
relu = ReLU()

out = relu.forward(x)
print("前向输出:", out)

dout = torch.ones_like(x)          # 假设上游梯度全为 1
d_input = relu.backward(dout)
print("反向梯度:", d_input)
# 可以看到 x <= 0 的位置梯度都被置 0，只有 x > 0 的位置梯度被原样传递

**练习 14**：手写 Sigmoid 的反向传播

Sigmoid 函数 $f(x)=\frac{1}{1+e^{-x}}$，其导数为 $f'(x)=f(x)(1-f(x))$。

1. 实现 `Sigmoid` 类：`forward` 保存输出 `out`；`backward` 按 `dout * (1 - out) * out` 计算
2. 输入 `x = torch.tensor([-2., -1., 0., 1., 2.])`（`requires_grad=True`），做前向与反向计算
3. 用 `torch.autograd`（对 `out.sum()` 调用 `backward()`）验证手写反向传播的梯度是否正确

In [ ]:
# 练习 14：手写 Sigmoid 的反向传播

# 练习 14：手写 Sigmoid 的反向传播

class Sigmoid:
    def __init__(self):
        self.out = None

    def forward(self, x):
        out = 1 / (1 + torch.exp(-x))
        self.out = out
        return out

    def backward(self, dout):
        # f'(x) = f(x)(1 - f(x))
        return dout * (1.0 - self.out) * self.out


x = torch.tensor([-2., -1., 0., 1., 2.], requires_grad=True)
sigmoid = Sigmoid()

out = sigmoid.forward(x)
print("前向输出:", out)

# 用自动微分验证：对 sum(out) 求导，相当于上游梯度为全 1
out.sum().backward()
print("autograd 梯度:", x.grad)

# 手写反向传播（上游梯度为全 1）
d_manual = sigmoid.backward(torch.ones_like(out))
print("手写梯度     :", d_manual)
print("两者是否一致 :", torch.allclose(d_manual, x.grad))

**练习 15**：全连接层的反向传播

对于全连接层 $Y = XW + B$（$X$ 形状 $N \times m$，$W$ 形状 $m \times n$），记上游梯度 $E=\frac{\partial L}{\partial Y}$，则

$$\frac{\partial L}{\partial X}=E \cdot W^{T}, \qquad \frac{\partial L}{\partial W}=X^{T} \cdot E$$

1. 用 `torch.manual_seed(42)` 生成 `X = torch.randn(4, 3, requires_grad=True)`、
   `W = torch.randn(3, 2, requires_grad=True)`、`B = torch.randn(2, requires_grad=True)`
2. 计算 `Y = X @ W + B`
3. 给定 `E = torch.randn(4, 2)`，手写反向传播：`dX = E @ W.T`、`dW = X.T @ E`、`dB = E.sum(dim=0)`
4. 用 `Y.backward(E)` 自动微分的结果验证手写梯度

In [ ]:
# 练习 15：全连接层的反向传播

# 练习 15：全连接层的反向传播

torch.manual_seed(42)

N, m, n = 4, 3, 2
X = torch.randn(N, m, requires_grad=True)   # N x m
W = torch.randn(m, n, requires_grad=True)   # m x n
B = torch.randn(n, requires_grad=True)      # n

# 1. 前向传播
Y = X @ W + B

# 2. 假设上游梯度 E = dL/dY
E = torch.randn(N, n)

# 3. 手写反向传播
dX_manual = E @ W.T             # dL/dX = E · W^T   -> (N x m)
dW_manual = X.T @ E             # dL/dW = X^T · E   -> (m x n)
dB_manual = E.sum(dim=0)        # dL/dB = E 按行求和 -> (n,)

# 4. 自动微分验证
Y.backward(E)
print("dX 是否一致:", torch.allclose(dX_manual, X.grad))
print("dW 是否一致:", torch.allclose(dW_manual, W.grad))
print("dB 是否一致:", torch.allclose(dB_manual, B.grad))

## 4.6 PyTorch 的自动微分模块

**练习 16**：计算图的梯度计算与叶子节点

考虑最简单的单层神经网络：

1. 定义 `x = torch.tensor(10.0)`、`y = torch.tensor(3.0)`、
   `w = torch.rand(1, 1, requires_grad=True)`、`b = torch.rand(1, 1, requires_grad=True)`
2. 前向传播 `z = w * x + b`，损失 `loss_value = nn.MSELoss()(z, y)`
3. 反向传播 `loss_value.backward()`，打印 `w.grad`、`b.grad`
4. 用 `is_leaf` 打印 `x`、`y`、`w`、`b`、`z`、`loss_value` 是否为叶子节点
5. 验证：$\frac{\partial L}{\partial w}=2(wx+b-y)x$，$\frac{\partial L}{\partial b}=2(wx+b-y)$

In [ ]:
# 练习 16：计算图的梯度计算与叶子节点

# 练习 16：计算图的梯度计算与叶子节点

torch.manual_seed(42)

# 1. 定义输入、目标值与可训练参数
x = torch.tensor(10.0)
y = torch.tensor(3.0)
w = torch.rand(1, 1, requires_grad=True)
b = torch.rand(1, 1, requires_grad=True)

# 2. 前向传播 + 计算损失
z = w * x + b
loss_fn = nn.MSELoss()
loss_value = loss_fn(z, y)
print("预测 z =", z.item(), " 真实 y =", y.item())

# 3. 反向传播
loss_value.backward()
print("w.grad =", w.grad)
print("b.grad =", b.grad)

# 4. 判断叶子节点
for name, tensor in [("x", x), ("y", y), ("w", w), ("b", b), ("z", z), ("loss_value", loss_value)]:
    print(f"{name}.is_leaf = {tensor.is_leaf}")

# 5. 手动验证：L = (z - y)^2，dL/dz = 2(z - y)，dz/dw = x，dz/db = 1
diff = 2 * (z - y)
print("手动 dw =", (diff * x).detach(), " 手动 db =", diff.detach())

**练习 17**：使用 `detach` 从计算图中分离张量

1. `x = torch.ones(2, 2, requires_grad=True)`
2. `y = x * x`，用 `u = y.detach()` 分离得到 `u`（`u` 与原变量值相同，但不再携带计算图信息）
3. `z = u * x`，执行 `z.sum().backward()`
4. 验证 `x.grad == u`（因为 `u` 被当作常数，$\frac{\partial z}{\partial x}=u$）
5. 思考：如果不使用 `detach`，`z = y * x = x^3`，梯度应该是多少？

In [ ]:
# 练习 17：使用 detach 从计算图中分离张量

# 练习 17：使用 detach 从计算图中分离张量

x = torch.ones(2, 2, requires_grad=True)
y = x * x

# 分离 y，得到的新变量 u 不携带计算图信息，梯度不会流经 u 传回
u = y.detach()
z = u * x

z.sum().backward()

print("x.grad:\n", x.grad)
print("u:\n", u)
print("x.grad == u :", torch.allclose(x.grad, u))

# 5. 若不用 detach，z = y * x = x^3，dz/dx = 3x^2 = 3（此时 x 全为 1）。
#    这说明 detach 后的变量在反向传播时会被当作常数处理。

**练习 18**：神经网络的一次完整训练迭代

实现 `Model`（继承 `nn.Module`），包含一个全连接层 `nn.Linear(5, 3)`：

- 权重 `weight` 初始化为下面矩阵的转置：
  - 第 1 行 `[0.1, 0.2, 0.3]`
  - 第 2 行 `[0.4, 0.5, 0.6]`
  - 第 3 行 `[0.7, 0.8, 0.9]`
  - 第 4 行 `[1.0, 1.1, 1.2]`
  - 第 5 行 `[1.3, 1.4, 1.5]`
- 偏置 `bias` 初始化为 `[1.0, 2.0, 3.0]`

然后：

1. 用输入 `X = torch.tensor([[1, 2, 3, 4, 5], [6, 7, 8, 9, 10]], dtype=torch.float)`、
   目标 `target = torch.zeros(2, 3)` 做前向传播
2. 用 `nn.MSELoss()` 计算损失并调用 `backward()`
3. 用 `optim.SGD(model.parameters(), lr=1)` 创建优化器，依次调用 `step()` 与 `zero_grad()`
4. 打印更新后的参数

In [ ]:
# 练习 18：神经网络的一次完整训练迭代

# 练习 18：神经网络的一次完整训练迭代

import torch
from torch import nn, optim


class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.linear1 = nn.Linear(5, 3)
        # 初始化权重（注意要转置，nn.Linear 的 weight 形状为 (out_features, in_features)）
        self.linear1.weight.data = torch.tensor([
            [0.1, 0.2, 0.3],
            [0.4, 0.5, 0.6],
            [0.7, 0.8, 0.9],
            [1.0, 1.1, 1.2],
            [1.3, 1.4, 1.5],
        ]).T
        # 初始化偏置
        self.linear1.bias.data = torch.tensor([1.0, 2.0, 3.0])

    def forward(self, x):
        return self.linear1(x)


model = Model()
X = torch.tensor([[1, 2, 3, 4, 5], [6, 7, 8, 9, 10]], dtype=torch.float)
target = torch.tensor([[0, 0, 0], [0, 0, 0]], dtype=torch.float)

# 前向传播 -> 计算损失 -> 反向传播
output = model(X)
loss_fn = nn.MSELoss()
loss_value = loss_fn(output, target)
loss_value.backward()

# 优化器更新参数并清空梯度
optimizer = optim.SGD(model.parameters(), lr=1)
optimizer.step()
optimizer.zero_grad()

print("损失:", loss_value.item())
for name, param in model.state_dict().items():
    print(name)
    print(param)

## 4.7 应用案例

**练习 19**：线性回归完整训练流程

用 PyTorch 训练一个线性回归模型，完整走一遍「准备数据 → 构建模型 → 定义损失函数与优化器 → 模型训练」四个步骤：

1. 准备数据：`X = torch.randn(100, 1)`，真实参数 `w = 2.5`、`b = 5.2`，
   噪声 `noise = torch.randn(100, 1) * 0.1`，目标 `y = w * X + b + noise`；
   用 `TensorDataset` 与 `DataLoader(batch_size=10, shuffle=True)` 封装
2. 构建模型：`nn.Linear(in_features=1, out_features=1)`
3. 定义损失函数与优化器：`nn.MSELoss()` 与 `optim.SGD(model.parameters(), lr=1e-3)`
4. 模型训练：训练 1000 个 epoch，每个 batch 执行「预测 → 计算损失 → `zero_grad()` → `backward()` → `step()`」，
   记录每个 epoch 的平均损失
5. 打印训练后的 `weight` 与 `bias`，与真实值 2.5、5.2 对比
6. 用 `matplotlib.pyplot` 画出损失随 epoch 变化的曲线

In [ ]:
# 练习 19：线性回归完整训练流程

# 练习 19：线性回归完整训练流程

import matplotlib.pyplot as plt

# 1. 准备数据
X = torch.randn(100, 1)                 # 输入
w = torch.tensor([2.5])                 # 真实权重
b = torch.tensor([5.2])                 # 真实偏置
noise = torch.randn(100, 1) * 0.1       # 噪声
y = w * X + b + noise                   # 目标

dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=10, shuffle=True)

# 2. 构建模型
model = nn.Linear(in_features=1, out_features=1)

# 3. 损失函数与优化器
loss_fn = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=1e-3)

# 4. 模型训练
loss_list = []
for epoch in range(1000):
    total_loss = 0.0
    train_num = 0
    for x_train, y_train in dataloader:
        y_pred = model(x_train)             # 前向传播
        loss_value = loss_fn(y_pred, y_train)  # 计算损失
        optimizer.zero_grad()               # 梯度清零
        loss_value.backward()               # 反向传播
        optimizer.step()                    # 更新参数
        total_loss += loss_value.item()
        train_num += len(y_train)
    loss_list.append(total_loss / train_num)

# 5. 打印训练结果
print("训练后的 weight:", model.weight.item(), "（真实值 2.5）")
print("训练后的 bias  :", model.bias.item(), "（真实值 5.2）")

# 6. 绘制损失曲线
plt.plot(loss_list)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Linear Regression Training Loss")
plt.show()

**练习 20**：用真实数据做线性回归（房价预测）

使用 `../../data/house_prices.csv`，用房屋面积 `GrLivArea` 预测房价 `SalePrice`：

1. 用 `pd.read_csv` 读取数据，取特征列 `GrLivArea` 与目标列 `SalePrice`（均为数值列）
2. 用 `train_test_split(X, y, test_size=0.2, random_state=42)` 划分训练集与测试集
3. 用 `StandardScaler` 对特征与目标分别做标准化（训练集 `fit_transform`，测试集 `transform`），
   再转成 `float32` 张量
4. 用 `TensorDataset` + `DataLoader(batch_size=64, shuffle=True)` 封装训练集
5. 用 `nn.Linear(1, 1)` + `nn.MSELoss()` + `optim.SGD(lr=0.01)` 训练若干 epoch
6. 在测试集上计算 MSE（标准化尺度下），并打印学到的权重与偏置

提示：房价数值较大（十万量级），直接训练容易发散，因此务必先做标准化。

In [ ]:
# 练习 20：用真实房价数据做线性回归

# 练习 20：用真实数据做线性回归（房价预测）

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. 读取数据
data = pd.read_csv("../../data/house_prices.csv")
X = data[["GrLivArea"]].values.astype("float32")     # 特征：地上居住面积
y = data[["SalePrice"]].values.astype("float32")     # 目标：房价
print("数据形状:", X.shape, y.shape)

# 2. 划分训练集与测试集
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. 标准化（特征与目标都要做）
x_scaler = StandardScaler()
y_scaler = StandardScaler()
x_train = x_scaler.fit_transform(x_train)
x_test = x_scaler.transform(x_test)
y_train = y_scaler.fit_transform(y_train)
y_test = y_scaler.transform(y_test)

x_train = torch.tensor(x_train).float()
x_test = torch.tensor(x_test).float()
y_train = torch.tensor(y_train).float()
y_test = torch.tensor(y_test).float()

# 4. 封装数据集与数据加载器
train_dataset = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# 5. 模型、损失函数与优化器
model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

for epoch in range(100):
    for x_batch, y_batch in train_loader:
        y_pred = model(x_batch)
        loss_value = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss_value.backward()
        optimizer.step()

# 6. 在测试集上评估
model.eval()
with torch.no_grad():
    test_pred = model(x_test)
    test_loss = loss_fn(test_pred, y_test)

print("测试集 MSE（标准化尺度）:", test_loss.item())
print("学到的权重:", model.weight.item(), " 偏置:", model.bias.item())
# 标准化尺度下权重约等于 GrLivArea 与 SalePrice 的相关系数，说明两者正相关

**练习 21**：编写手写数字识别的数据读取函数 `get_data()`

读取 `../../data/train.csv`（第 1 列为标签，之后 784 列为 28×28 像素亮度值），
编写函数返回 `x_train, x_test, y_train, y_test`：

1. 用 `pd.read_csv` 加载数据集
2. `X = data.drop("label", axis=1)` 作为特征，`y = data["label"]` 作为标签
3. 用 `train_test_split(X, y, test_size=0.3, random_state=42)` 划分训练集与测试集
4. 用 `MinMaxScaler` 归一化（训练集 `fit_transform`，测试集 `transform`）
5. 特征转成 `float` 张量，标签转成整型张量后返回

调用该函数并打印各个变量的形状。

In [ ]:
# 练习 21：编写手写数字识别的数据读取函数 get_data()

# 练习 21：编写手写数字识别的数据读取函数 get_data()

from sklearn.model_selection import train_test_split   # 划分数据集
from sklearn.preprocessing import MinMaxScaler         # 归一化 Scaler


def get_data():
    # 1. 加载数据集
    data = pd.read_csv("../../data/train.csv")
    # 2. 分离特征与标签
    X = data.drop("label", axis=1)   # 特征（784 列像素）
    y = data["label"]                # 标签（0~9）
    # 3. 划分训练集与测试集
    x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    # 4. 归一化
    scaler = MinMaxScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)
    # 5. 转成张量
    x_train = torch.tensor(x_train).float()
    x_test = torch.tensor(x_test).float()
    y_train = torch.tensor(y_train.values)
    y_test = torch.tensor(y_test.values)
    return x_train, x_test, y_train, y_test


x_train, x_test, y_train, y_test = get_data()
print("x_train:", tuple(x_train.shape), x_train.dtype)
print("x_test :", tuple(x_test.shape), x_test.dtype)
print("y_train:", tuple(y_train.shape), y_train.dtype)
print("y_test :", tuple(y_test.shape), y_test.dtype)

**练习 22**：手写数字识别的完整训练与验证

搭建三层神经网络（输入 784 → 隐藏层 50 → 隐藏层 100 → 输出 10，隐藏层激活函数为 ReLU），
在 `train.csv` 上完整训练并验证：

1. 复用练习 21 的 `get_data()` 读入数据，构造 `TensorDataset` 与 `DataLoader`
   （训练集 `batch_size=64, shuffle=True`，测试集 `batch_size=64`）
2. 模型用 `nn.Sequential` 搭建；损失函数 `nn.CrossEntropyLoss()`；优化器 `optim.SGD(model.parameters(), lr=0.1)`
3. 训练若干 epoch（如 5），每个 epoch 内：
   - `model.train()`：遍历 `train_loader`，累加损失并统计预测正确的样本数（用 `output.argmax(dim=1)`）
   - `model.eval()` + `with torch.no_grad():`：遍历测试集的 `DataLoader`，累加损失与正确数
   - 打印该 epoch 的训练/验证损失与准确率
4. 观察「训练集准确率持续上升、验证集准确率趋于稳定」的过程

In [ ]:
# 练习 22：手写数字识别的完整训练与验证

# 练习 22：手写数字识别的完整训练与验证

# 1. 读入数据
x_train, x_test, y_train, y_test = get_data()

# 2. 创建模型
model = nn.Sequential(
    nn.Linear(784, 50),
    nn.ReLU(),
    nn.Linear(50, 100),
    nn.ReLU(),
    nn.Linear(100, 10),
)

# 3. 设置超参数
lr = 0.1
batch_size = 64
epochs = 5

# 4. 构建 DataSet 和 DataLoader
train_dataset = TensorDataset(x_train, y_train)
test_dataset = TensorDataset(x_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# 5. 定义损失函数和优化器
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=lr)

# 6. 定义设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 7. 模型训练和验证
for epoch in range(epochs):
    # 7.1 训练
    model.train()
    train_loss_total = 0.0
    train_acc_cnt = 0
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        output = model(x_batch)                  # 前向传播
        loss_value = loss_fn(output, y_batch)    # 计算损失
        optimizer.zero_grad()                    # 梯度清零
        loss_value.backward()                    # 反向传播
        optimizer.step()                         # 更新参数

        train_loss_total += loss_value.item() * x_batch.shape[0]
        y_pred = output.argmax(dim=1)
        train_acc_cnt += y_pred.eq(y_batch).sum().item()

    train_loss = train_loss_total / len(train_dataset)
    train_acc = train_acc_cnt / len(train_dataset)

    # 7.2 验证
    model.eval()
    test_loss_total = 0.0
    test_acc_cnt = 0
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            output = model(x_batch)
            loss_value = loss_fn(output, y_batch)
            test_loss_total += loss_value.item() * x_batch.shape[0]
            y_pred = output.argmax(dim=1)
            test_acc_cnt += y_pred.eq(y_batch).sum().item()

    test_loss = test_loss_total / len(test_dataset)
    test_acc = test_acc_cnt / len(test_dataset)

    print(f"epoch {epoch + 1}/{epochs} | train loss:{train_loss:.4f} acc:{train_acc:.4f} "
          f"| val loss:{test_loss:.4f} acc:{test_acc:.4f}")